#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, regexp_replace

In [0]:

catlog = "`abhi-de-ete-project-catlog`"
tbl_name = "erp_cust_az12"

bronze_table_path = f"{catlog}.bronze.{tbl_name}"


# Read Bronze table

In [0]:
df = spark.table(bronze_table_path)

#Silver Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Customer ID Cleanup

In [0]:
df.display()

In [0]:
df.select(col('CID'),F.length(col("CID"))).collect()[0]

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.regexp_replace(col('cid'), "NAS", ""))
     .otherwise(col("cid"))
)


In [0]:
df.show()

##Birthdate Validation

In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

##Gender Normalization

In [0]:
df.select('gen').distinct().display()

In [0]:
female_gender_abbr = ["F", "FEMALE"]
male_gender_abbr = ["M", "MALE"]

df = df.withColumn('gen',
                   F.when(F.upper(col("GEN")).isin(female_gender_abbr), "Female")
                    .when(F.upper(col("GEN")).isin(male_gender_abbr), "Male")
                    .otherwise("n/a")
)

In [0]:
df.select('gen').distinct().display()

## Renaming Columns

In [0]:

RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{catlog}.silver.erp_customers")

## Sanity checks of silver table

In [0]:

spark.sql(f"SELECT * FROM {catlog}.silver.erp_customers LIMIT 10").display()